# Padrón electoral vs población censal — Sonora

¿Qué proporción de la población está empadronada? La pregunta parece
simple pero tiene una trampa: **el padrón electoral solo contiene
personas en edad de votar con credencial vigente**, mientras que el
censo cuenta a toda la población. Comparar `pad` contra población
total produce razones artificialmente bajas (~60–65 %) que **no**
miden cobertura electoral.

Esta libreta calcula ambos denominadores:

- `pad / pob_total` — referencia general, subestima cobertura real.
- `pad / pob_18+` — aproximación a la cobertura del padrón sobre la
  población en edad de votar (aún imperfecta: el padrón excluye
  ciudadanos sin credencial, incluye fallecidos no depurados, etc.).

**Fuentes**: `pad`/`ln` de cartografía INE (2019–2025) · `ln` del
conteo censal electoral (2009–2024) · población INEGI 2010/2020 ·
CONAPO 1990–2040 (denominador interpolable).

**Limitación clave**: no se localizó una descarga directa del
*Censos y Conteos a Escalas Geoelectorales* (ECGE/EGIE) de INEGI a
nivel **sección**, así que los denominadores censales llegan hasta
municipio; el nivel sección solo permite comparar magnitudes
electorales entre sí (`ln`/`pad`).

In [1]:
import json

import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from vothmo.config import FOLDER_DATA

PROCESSED = FOLDER_DATA / "processed"
MAP_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
CENTRO_SONORA = {"lat": 29.3, "lon": -110.9}
ANIO = 2025            # año de cartografía para los mapas

conapo = pd.read_csv(PROCESSED / "conapo_sonora.csv")
inegi = pd.read_csv(PROCESSED / "censo_inegi_municipios.csv")
piv = pd.read_csv(PROCESSED / "seccion_distrito_por_anio.csv")

pad_sec = pd.concat([
    gpd.read_file(PROCESSED / f"secciones_{y}.gpkg", ignore_geometry=True)
      .assign(anio=y)
    for y in MAP_YEARS
], ignore_index=True)[["anio", "seccion", "municipio", "mpiocve", "distrito_l", "pad", "ln"]]

conteos = pd.read_csv(PROCESSED / "conteos_sonora.csv.gz",
                      usecols=["feleccion", "mpiocve", "mpionom",
                               "seccion", "del", "ln"],
                      low_memory=False)
conteos["anio_elec"] = conteos["feleccion"].str[-4:].astype(int)

# La cartografía, los conteos del INE y las capas `*_simp.gpkg`
# usan la clave municipal del INE; `mun_ine_inegi.csv` la traduce
# al catálogo INEGI. Las capas secciones_*.gpkg ya traen `mpiocve`.
munmap = pd.read_csv(PROCESSED / "mun_ine_inegi.csv")

mun_geo = gpd.read_file(PROCESSED / "municipios_simp.gpkg")
dl_geo = gpd.read_file(PROCESSED / f"distritos_simp_{ANIO}.gpkg")

def geojson_ligero(gdf: gpd.GeoDataFrame) -> dict:
    """GeoJSON con solo geometría e id (los atributos van en los datos)."""
    feats = json.loads(gdf.to_json())["features"]
    for i, f in enumerate(feats):
        f["id"] = i
        f["properties"] = {}
    return {"type": "FeatureCollection", "features": feats}


## 1. Nivel sección: solo magnitudes electorales

Sin denominador censal por sección (ECGE no disponible por descarga
directa), lo que sí se puede medir es la **razón `ln`/`pad`** por
sección y año — cobertura de la lista nominal sobre el padrón — y su
evolución. Valores sistemáticamente bajos indican credenciales no
entregadas o padrón sin depurar.

In [2]:
sec = pad_sec.copy()
sec["razon_ln_pad"] = sec.ln / sec.pad

resumen_sec = (sec.groupby("anio")
               .agg(pad=("pad", "sum"), ln=("ln", "sum")))
resumen_sec["razon_ln_pad"] = resumen_sec.ln / resumen_sec.pad

fig = px.line(resumen_sec.reset_index(), x="anio", y="razon_ln_pad",
              markers=True,
              title="Razón lista nominal / padrón, Sonora (todos los años)")
fig.update_layout(height=380, yaxis_tickformat=".1%", yaxis_title="ln / pad")
fig.show()

print("Secciones con razon ln/pad < 0.9 o > 1.01 en", ANIO, ":")
ext = sec[(sec.anio == ANIO) & ((sec.razon_ln_pad < 0.9) | (sec.razon_ln_pad > 1.01))]
print(f"{len(ext)} de {len(sec[sec.anio == ANIO])} secciones "
      f"({len(ext)/len(sec[sec.anio == ANIO]):.1%})")
ext.sort_values("razon_ln_pad").head(10)


Secciones con razon ln/pad < 0.9 o > 1.01 en 2025 :
0 de 1667 secciones (0.0%)


,anio,seccion,municipio,mpiocve,distrito_l,pad,ln,razon_ln_pad


## 2. Nivel municipio: padrón vs población

Para cada año de cartografía se confronta `pad` municipal con la
población total y 18+ de CONAPO en el mismo año, y con los censos
INEGI 2010/2020 como contrapunto. Cambia `ANIO` para otros cortes.

> **Ojo con las claves**: la cartografía y el conteo electoral del INE
> numeran los municipios con su propio catálogo (distinto del
> `mpiocve` de INEGI). La traducción `mun_ine → mpiocve` viene de
> `mun_ine_inegi.csv`, generada en `00_preparacion_datos`.

In [3]:
pad_mun = pad_sec.groupby(["anio", "mpiocve"]).pad.sum().reset_index()

conapo_mun = (conapo.groupby(["mpiocve", "nom_mun", "anio"])
              .agg(pob=("pob_total", "sum"), pob18=("pob_18mas", "sum"))
              .reset_index())

cmp_mun = (pad_mun.merge(conapo_mun, on=["mpiocve", "anio"], how="left")
           .assign(razon_total=lambda d: d.pad / d.pob,
                   razon_18=lambda d: d.pad / d.pob18))

estado = (cmp_mun.groupby("anio")
          .agg(pad=("pad", "sum"), pob=("pob", "sum"), pob18=("pob18", "sum")))
estado["razon_total"] = estado.pad / estado.pob
estado["razon_18"] = estado.pad / estado.pob18

fig = go.Figure()
fig.add_scatter(x=estado.index, y=estado.razon_18,
                name="pad / pob 18+ (CONAPO)", mode="lines+markers")
fig.add_scatter(x=estado.index, y=estado.razon_total,
                name="pad / pob total (CONAPO)", mode="lines+markers")
fig.update_layout(title="Proporción empadronada, Sonora",
                  yaxis_tickformat=".1%", yaxis_title="proporción",
                  height=400)
fig.show()
estado.round(3)


,pad,pob,pob18,razon_total,razon_18
anio,,,,,
2019,2032271,2981674,2056413.2,0.682,0.988
2020,2108953,3002286,2085823.0,0.702,1.011
2021,2191210,3024147,2117438.6,0.725,1.035
2022,2140362,3051188,2154248.4,0.701,0.994
2023,2221477,3081734,2194597.2,0.721,1.012
2024,2295226,3111704,2234350.0,0.738,1.027
2025,2317487,3140762,2273140.0,0.738,1.020


**Contraste con INEGI**: en años censales exactos
(2010, 2020) la razón se puede medir con denominador censal real. Como
la cartografía `pad` empieza en 2019, para 2010 se usa el `ln` del
conteo electoral más cercano (2012, corte de marzo) — una
aproximación con sesgo temporal explícito.

In [4]:
# 2020: pad 2020 vs censo 2020 (ambos del mismo año)
r2020 = (pad_mun[pad_mun.anio == 2020]
         .merge(inegi[inegi.anio == 2020], on=["mpiocve"])
         .assign(razon_18=lambda d: d.pad / d.p_18ymas,
                 razon_total=lambda d: d.pad / d.pobtot))

# 2010: ln del conteo electoral 2012 vs censo 2010 (sesgo temporal ~2 años)
ln2012 = (conteos[conteos.anio_elec == 2012]
          .rename(columns={"mpiocve": "mun_ine"})
          .groupby("mun_ine").ln.sum().rename("ln_2012").reset_index()
          .merge(munmap[["mun_ine", "mpiocve"]], on="mun_ine")
          [["mpiocve", "ln_2012"]])
r2010 = (inegi[inegi.anio == 2010].merge(ln2012, on="mpiocve")
         .assign(razon_18=lambda d: d.ln_2012 / d.p_18ymas,
                 razon_total=lambda d: d.ln_2012 / d.pobtot))

resumen_inegi = pd.DataFrame({
    "2010 (ln 2012 / censo 2010)": [
        r2010.ln_2012.sum() / r2010.pobtot.sum(),
        r2010.ln_2012.sum() / r2010.p_18ymas.sum()],
    "2020 (pad 2020 / censo 2020)": [
        r2020.pad.sum() / r2020.pobtot.sum(),
        r2020.pad.sum() / r2020.p_18ymas.sum()],
}, index=["razon vs pob total", "razon vs pob 18+"]).round(3)
resumen_inegi


,2010 (ln 2012 / censo 2010),2020 (pad 2020 / censo 2020)
razon vs pob total,0.646,0.716
razon vs pob 18+,0.998,1.021


## 3. Mapa municipal de la razón padrón / 18+

Cambia `ANIO` para explorar otros cortes cartográficos. Valores > 1
pueden indicar padrón inflado (sin depurar), inmigración posterior a
la estimación CONAPO, o subestimación del denominador.

In [5]:
sel = (cmp_mun[cmp_mun.anio == ANIO]
       [["mpiocve", "nom_mun", "pad", "pob", "pob18",
         "razon_total", "razon_18"]])
g = (mun_geo.drop(columns=["ln_2024", "pad_2024"])
     .reset_index(drop=True)
     .merge(sel.drop(columns="nom_mun"), on="mpiocve", how="left"))

fig = px.choropleth_map(
    g, geojson=geojson_ligero(g), locations=g.index,
    color="razon_18", hover_name="nom_mun",
    hover_data={"pad": ":,", "pob18": ":,.0f",
                "razon_total": ".1%"},
    color_continuous_scale="RdYlGn_r", range_color=[0.7, 1.15],
    center=CENTRO_SONORA, zoom=5, opacity=0.75,
    title=f"pad / población 18+ por municipio, {ANIO}")
fig.update_layout(height=600, coloraxis_colorbar_tickformat=".0%")
fig.show()

print("Municipios con razón más alta:")
display_cols = ["nom_mun", "pad", "pob18", "razon_18", "razon_total"]
print(sel.sort_values("razon_18", ascending=False)[display_cols].head(8)
      .to_string(index=False))
print("\nMunicipios con razón más baja:")
print(sel.sort_values("razon_18")[display_cols].head(8).to_string(index=False))


Municipios con razón más alta:
            nom_mun  pad  pob18  razon_18  razon_total
             Onavas  761  272.0  2.797794     2.180516
    Villa Pesqueira 1900  768.6  2.472027     1.989529
San Felipe de Jesús  557  259.8  2.143957     1.542936
        Divisaderos 1192  603.6  1.974818     1.457213
            Mazatán 1522  810.8  1.877158     1.515936
        La Colorada 2622 1400.4  1.872322     1.468085
           Bacanora 1055  593.6  1.777291     1.391821
         San Javier  623  360.0  1.730556     1.160149

Municipios con razón más baja:
                 nom_mun    pad    pob18  razon_18  razon_total
                 Oquitoa    441    532.0  0.828947     0.640058
                   Sáric   1744   1993.8  0.874712     0.618440
              Hermosillo 693700 755412.0  0.918307     0.674219
                Tubutama   1329   1428.8  0.930151     0.700211
San Miguel de Horcasitas   7468   8017.4  0.931474     0.601385
               Pitiquito   6350   6666.4  0.952538     0.6

## 4. Nivel distrito local

El padrón se agrega a distrito directamente; la población 18+ se
reparte con pesos de padrón 2025 (igual que en `03_escenarios`), por
lo que la razón por distrito es una **aproximación**: los municipios
partidos entre distritos se distribuyen según su padrón, no según su
población censal real por sección.

In [6]:
w = (pad_sec[pad_sec.anio == ANIO]
     .groupby(["mpiocve", "distrito_l"]).pad.sum().reset_index())
w["peso"] = w.pad / w.groupby("mpiocve").pad.transform("sum")

pob18_mun = conapo_mun[conapo_mun.anio == ANIO][["mpiocve", "pob18"]]
dl = (w.merge(pob18_mun, on="mpiocve")
      .assign(pob18_dl=lambda d: d.pob18 * d.peso)
      .groupby("distrito_l")
      .agg(pad=("pad", "sum"), pob18=("pob18_dl", "sum"))
      .reset_index())
dl["razon_18"] = dl.pad / dl.pob18

gd = dl_geo.drop(columns=["ln", "pad"]).reset_index(drop=True).merge(dl, on="distrito_l", how="left")
fig = px.choropleth_map(
    gd, geojson=geojson_ligero(gd), locations=gd.index,
    color="razon_18", hover_name="distrito_l",
    hover_data={"pad": ":,", "pob18": ":,.0f"},
    color_continuous_scale="RdYlGn_r", range_color=[0.7, 1.15],
    center=CENTRO_SONORA, zoom=5, opacity=0.75,
    title=f"pad / población 18+ por distrito local, {ANIO} (aprox.)")
fig.update_layout(height=600, coloraxis_colorbar_tickformat=".0%")
fig.show()

fig = px.bar(dl.sort_values("razon_18"), x="distrito_l", y="razon_18",
             title="Razón pad/18+ por distrito local (ordenada)")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()
dl.sort_values("razon_18", ascending=False).round(3)


,distrito_l,pad,pob18,razon_18
1,2,118482,95937.806,1.235
0,1,127194,109375.994,1.163
17,18,114997,99924.000,1.151
20,21,108334,96823.994,1.119
6,7,116712,104579.600,1.116
4,5,116374,105932.057,1.099
3,4,106990,97409.943,1.098
13,14,90697,85574.900,1.060
12,13,102057,97243.900,1.049
19,20,93638,90709.754,1.032


## 5. Distribución de razones y valores atípicos

In [7]:
fig = px.histogram(cmp_mun[cmp_mun.anio == ANIO], x="razon_18",
                   nbins=25, marginal="box",
                   title=f"Distribución municipal de pad/18+, {ANIO}")
fig.update_layout(height=380, xaxis_tickformat=".0%")
fig.show()

# evolución de la razón estatal vs tiempo, todos los años de cartografía
evol = estado[["razon_total", "razon_18"]].reset_index()
print("Correlación razón 18+ municipal vs tamaño municipal (log pob):")
sel["log_pob"] = sel.pob.clip(lower=1).apply(lambda x: x)
import numpy as np
print(np.corrcoef(np.log10(sel.pob.clip(lower=1)), sel.razon_18).round(3)[0, 1])


Correlación razón 18+ municipal vs tamaño municipal (log pob):
-0.631


## 6. Caveats de interpretación

- **Fechas que no coinciden**: el `pad` cartográfico es un corte
  administrativo (típicamente del año de publicación), mientras CONAPO
  interpola entre censos y el censo es un día específico. Razones
  ligeramente >1 no prueban sobrepadrón.
- **`ln` del conteo electoral ≠ `pad` cartográfico**: la lista nominal
  descarta credenciales no entregadas; el padrón las incluye. En 2010
  se usó el `ln` de 2012 contra el censo 2010, un desfase de ~2 años
  señalado explícitamente.
- **El denominador municipal → distrito es proxy**: los municipios
  partidos se reparten por peso de padrón 2025, no por población
  seccional censada.
- **Sin fuente seccional**: hasta conseguir los datos del ECGE/INEGI
  (Estadísticas Censales a Escalas Geoelectorales) la comparación
  por sección queda limitada a magnitudes electorales internas.
- Una razón pad/18+ de ~0.95–1.05 es plausible; valores muy bajos
  suelen señalar subregistro o jóvenes sin credencial, valores muy
  altos señalan rezago de depuración o envejecimiento del padrón
  relativo a la estimación poblacional.